# 08 — Robustness Tests

Three independent tests validating the main quantile LP results from notebook 07.

| Test | Critique addressed | Expected result |
|------|-------------------|-----------------|
| A. Cross-asset placebo | "It's just crypto beta" | Short-horizon: ETH > placebos. Long-horizon: placebos catch up (illiquidity) |
| B. Block bootstrap | "SEs are wrong (serial correlation)" | 95% CI excludes zero at key horizons |
| C. Sensitivity | "Results depend on arbitrary thresholds" | Stable beta across specs |

In [4]:
# ── Setup (run this first, always) ──
import sys; sys.path.insert(0, "..")
from config import CFG, ECON_DIR, REPORTS_DIR
CFG.ensure_dirs()

import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm

print("Setup OK")

Setup OK


## Test A — Cross-Asset Placebo (orthogonalized shock)

We orthogonalize the liquidation shock by regressing `log_liq` on BTC returns,
then using the residual as our shock variable. This isolates the DeFi-specific
component of liquidations, removing the "market went down" signal.

We then run the same quantile LP on vol-normalized returns for ETH, BTC, XRP, DOGE.

In [6]:
# ── A1. Load all data + merge placebos ──

# Main panel
df = pd.read_parquet(CFG.FILES.econ_core_full, engine="pyarrow")
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date").reset_index(drop=True)

# Load placebo spot data
def load_spot(name):
    path = CFG.ROOT / "data" / "normalized" / "spot" / f"{name}_ccdata_1h.parquet"
    s = pd.read_parquet(path, engine="pyarrow")
    s["date"] = pd.to_datetime(s["date"], utc=True)
    return s[["date", "close"]].rename(columns={"close": f"close_{name}"})

df = df.merge(load_spot("xrp"), on="date", how="left")
df = df.merge(load_spot("doge"), on="date", how="left")

# Compute returns + vol-normalized returns for placebos
vol_window = CFG.ECON.vol_window  # 168h
for asset in ["xrp", "doge"]:
    df[f"ret_{asset}"] = np.log(df[f"close_{asset}"]).diff() * 100
    vol = df[f"ret_{asset}"].rolling(vol_window).std()
    df[f"ret_{asset}_std"] = df[f"ret_{asset}"] / vol.replace(0, np.nan)

print(f"Panel: {len(df):,} rows")
print(f"XRP valid: {df['ret_xrp_std'].notna().sum():,}")
print(f"DOGE valid: {df['ret_doge_std'].notna().sum():,}")

Panel: 41,328 rows
XRP valid: 41,160
DOGE valid: 41,160


In [9]:
# ── A2. Build estimation sample + orthogonalize shock + cumulative returns ──

test_horizons = [0, 1, 3, 6, 12, 24]
test_quantiles = [0.01, 0.05, 0.50]
controls = ["ret_btc_spot", "funding_rate", "basis_bps"]

# Assets: name → vol-normalized return column
assets = {
    "ETH": "ret_eth_std",
    "BTC": "ret_btc_std",
    "XRP": "ret_xrp_std",
    "DOGE": "ret_doge_std",
}

# Warmup: max of rolling windows + max horizon + buffer
warmup = max(vol_window, 720) + max(test_horizons) + 2
df_est = df.iloc[warmup:].copy()

# ── Orthogonalize shock: remove BTC-explained component ──
df_est["ret_btc_lag1"] = df_est["ret_btc_spot"].shift(1)
X_orth = sm.add_constant(df_est[["ret_btc_spot", "ret_btc_lag1"]].fillna(0))
y_orth = df_est["log_liq"].fillna(0)
mask_orth = X_orth.notna().all(axis=1) & y_orth.notna()
orth_model = OLS(y_orth[mask_orth], X_orth[mask_orth]).fit()
df_est["shock_orth"] = np.nan
df_est.loc[mask_orth, "shock_orth"] = orth_model.resid

# Shock = lagged orthogonalized residual
df_est["shock"] = df_est["shock_orth"].shift(1)

print(f"Orthogonalization R² = {orth_model.rsquared:.4f}")
print(f"  → {100*(1-orth_model.rsquared):.1f}% of liquidation variation is DeFi-specific")

# ── Cumulative vol-normalized returns for each asset × horizon ──
for asset_name, ret_col in assets.items():
    for h in test_horizons:
        if h == 0:
            df_est[f"cumret_{asset_name}_h{h}"] = df_est[ret_col]
        else:
            df_est[f"cumret_{asset_name}_h{h}"] = df_est[ret_col].rolling(h + 1).sum().shift(-h)

print(f"Estimation sample: {len(df_est):,} rows")

Orthogonalization R² = 0.0768
  → 92.3% of liquidation variation is DeFi-specific
Estimation sample: 40,582 rows


In [11]:
# ── A3. Estimate β(shock) for each asset × quantile × horizon ──

placebo_results = []

for asset_name, ret_col in assets.items():
    for tau in test_quantiles:
        for h in test_horizons:
            y_col = f"cumret_{asset_name}_h{h}"
            regressors = ["shock"] + controls

            mask = df_est[[y_col, "shock"] + controls].notna().all(axis=1)
            y = df_est.loc[mask, y_col]
            X = sm.add_constant(df_est.loc[mask, regressors].fillna(0))

            if len(y) < 500:
                continue

            try:
                model = QuantReg(y, X)
                res = model.fit(q=tau, vcov="robust", kernel="epa",
                                bandwidth="hsheather", max_iter=5000)
                placebo_results.append({
                    "asset": asset_name,
                    "tau": tau,
                    "h": h,
                    "beta_shock": res.params.get("shock", np.nan),
                    "se_shock": res.bse.get("shock", np.nan),
                    "pval_shock": res.pvalues.get("shock", np.nan),
                    "n_obs": int(res.nobs),
                })
            except Exception as e:
                print(f"  Warning: {asset_name} tau={tau} h={h}: {e}")

    print(f"  {asset_name} done")

df_placebo = pd.DataFrame(placebo_results)
print(f"Total estimates: {len(df_placebo)}")

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (5000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


  ETH done


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (5000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (5000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


  BTC done
  XRP done
  DOGE done
Total estimates: 72


In [35]:
# ── A4. Display results ──

print("=" * 65)
print("TEST A: beta(shock) at tau=0.01 — DeFi collateral exposure gradient")
print("(More negative = stronger amplification by DeFi liquidations)")
print("=" * 65)

pivot_01 = df_placebo[df_placebo["tau"] == 0.01].pivot_table(
    index="h", columns="asset", values="beta_shock"
)[["ETH", "BTC", "XRP", "DOGE"]]
print(pivot_01.round(4).to_string())

print("\n" + "=" * 65)
print("beta(shock) at tau=0.50 (median — should be ~0 for all)")
print("=" * 65)

pivot_50 = df_placebo[df_placebo["tau"] == 0.50].pivot_table(
    index="h", columns="asset", values="beta_shock"
)[["ETH", "BTC", "XRP", "DOGE"]]
print(pivot_50.round(4).to_string())

# Ratios at h=0 and h=12
print("\n--- Sensitivity ratios at tau=0.01 ---")
for h_check in [0, 3, 12]:
    if h_check in pivot_01.index:
        eth_b = pivot_01.loc[h_check, "ETH"]
        for asset in ["BTC", "XRP", "DOGE"]:
            other_b = pivot_01.loc[h_check, asset]
            if other_b != 0 and not np.isnan(other_b):
                ratio = eth_b / other_b
                print(f"  h={h_check:>2}: ETH/{asset} = {ratio:.2f}x")

TEST A: beta(shock) at tau=0.01 — DeFi collateral exposure gradient
(More negative = stronger amplification by DeFi liquidations)
asset     ETH     BTC     XRP    DOGE
h                                    
0     -0.0748 -0.0657 -0.0591 -0.0608
1     -0.1131 -0.1023 -0.1161 -0.0923
3     -0.1352 -0.1245 -0.1152 -0.1503
6     -0.1729 -0.1627 -0.2049 -0.2462
12    -0.2071 -0.1291 -0.2895 -0.2893
24    -0.1891 -0.1679 -0.3246 -0.2985

beta(shock) at tau=0.50 (median — should be ~0 for all)
asset     ETH     BTC     XRP    DOGE
h                                    
0      0.0015 -0.0002  0.0011  0.0033
1      0.0033 -0.0002  0.0063  0.0056
3      0.0052  0.0040  0.0153  0.0101
6      0.0138  0.0070  0.0194  0.0179
12     0.0320  0.0198  0.0354  0.0370
24     0.0267  0.0291  0.0404  0.0373

--- Sensitivity ratios at tau=0.01 ---
  h= 0: ETH/BTC = 1.14x
  h= 0: ETH/XRP = 1.27x
  h= 0: ETH/DOGE = 1.23x
  h= 3: ETH/BTC = 1.09x
  h= 3: ETH/XRP = 1.17x
  h= 3: ETH/DOGE = 0.90x
  h=12: ETH/BTC = 1

## Test B — Block Bootstrap Confidence Bands

Block bootstrap with 24h blocks preserves serial correlation structure.
Set `N_BOOT = 50` for testing, `N_BOOT = 1000` for the paper.

In [13]:
# ── B1. Block bootstrap function ──

def block_bootstrap_qlp(data, y_col, regressors, tau, n_boot, block_size=24):
    """Block bootstrap for quantile regression. Returns array of beta(shock)."""
    mask = data[[y_col] + regressors].notna().all(axis=1)
    clean = data.loc[mask].reset_index(drop=True)
    n = len(clean)
    n_blocks = n // block_size

    betas = []
    for b in range(n_boot):
        block_starts = np.random.randint(0, n - block_size, size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block_size) for s in block_starts])
        idx = idx[idx < n]

        boot = clean.iloc[idx]
        y = boot[y_col]
        X = sm.add_constant(boot[regressors].fillna(0))

        try:
            res = QuantReg(y, X).fit(q=tau, max_iter=3000)
            betas.append(res.params.get("shock", np.nan))
        except:
            pass

        if (b + 1) % 50 == 0:
            print(f"    {b+1}/{n_boot}", end="\r")

    return np.array([b for b in betas if not np.isnan(b)])

print("Bootstrap function defined")

Bootstrap function defined


In [15]:
# ── B2. Run bootstrap ──

N_BOOT = 1000  # ← Set to 1000 for paper

boot_horizons = [0, 3, 6, 12, 24]
boot_regressors = ["shock", "shock_x_oi", "oi_high", "funding_rate", "basis_bps"]

# Ensure interaction variable exists
df_est["shock_x_oi"] = df_est["shock"].fillna(0) * df_est["oi_high"]

# Ensure cumret columns exist for ETH
for h in boot_horizons:
    col = f"cumret_ETH_h{h}"
    if col not in df_est.columns:
        if h == 0:
            df_est[col] = df_est["ret_eth_std"]
        else:
            df_est[col] = df_est["ret_eth_std"].rolling(h + 1).sum().shift(-h)

boot_results = {}
for h in boot_horizons:
    print(f"  h={h}...")
    y_col = f"cumret_ETH_h{h}"
    betas = block_bootstrap_qlp(df_est, y_col, boot_regressors, tau=0.01,
                                 n_boot=N_BOOT, block_size=CFG.ECON.block_boot_size)
    boot_results[h] = {
        "mean": float(np.mean(betas)),
        "median": float(np.median(betas)),
        "ci_lo": float(np.percentile(betas, 2.5)),
        "ci_hi": float(np.percentile(betas, 97.5)),
        "n_success": len(betas),
        "pct_negative": float(100 * np.mean(betas < 0)),
    }

print("\n" + "=" * 55)
print(f"TEST B: Block Bootstrap — beta(shock) at tau=0.01")
print(f"({N_BOOT} replications, block size = {CFG.ECON.block_boot_size}h)")
print("=" * 55)
print(f"{'h':>3}  {'mean':>8}  {'median':>8}  {'CI 2.5%':>8}  {'CI 97.5%':>8}  {'% neg':>6}")
print("-" * 50)
for h, r in boot_results.items():
    print(f"{h:>3}  {r['mean']:>8.4f}  {r['median']:>8.4f}  {r['ci_lo']:>8.4f}  {r['ci_hi']:>8.4f}  {r['pct_negative']:>5.1f}%")

  h=0...


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


...

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regr

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (3000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


    1000/1000
TEST B: Block Bootstrap — beta(shock) at tau=0.01
(1000 replications, block size = 24h)
  h      mean    median   CI 2.5%  CI 97.5%   % neg
--------------------------------------------------
  0   -0.0832   -0.0847   -0.1203   -0.0408  100.0%
  3   -0.1280   -0.1253   -0.2498   -0.0372   99.8%
  6   -0.1269   -0.1142   -0.2964    0.0022   97.2%
 12   -0.2066   -0.1957   -0.5046   -0.0333   98.8%
 24   -0.1209   -0.1158   -0.3420    0.0644   89.3%


## Test C — Sensitivity to Specification Choices

Re-estimate the main beta(0.01, h=0) under alternative specifications:
OI regime at P70/P80/P90, and collateral_seized instead of debt_repaid.

In [17]:
# ── C1. Sensitivity analysis ──

# Reload clean panel (independent of Test A/B state)
df_c = pd.read_parquet(CFG.FILES.econ_core_full, engine="pyarrow")
df_c["date"] = pd.to_datetime(df_c["date"], utc=True)
df_c = df_c.sort_values("date").reset_index(drop=True)

tau = 0.01
warmup_c = 750
sensitivity_results = []

# ── Grid of OI thresholds ──
for oi_pctile in [70, 80, 90]:
    oi_flag = (df_c["oi"].rolling(720).rank(pct=True) > (oi_pctile / 100)).astype(int)
    shock = df_c["log_liq"].shift(1)
    interaction = shock * oi_flag

    y = df_c["ret_eth_perp"].iloc[warmup_c:]
    X = sm.add_constant(pd.DataFrame({
        "shock": shock.iloc[warmup_c:],
        "shock_x_oi": interaction.iloc[warmup_c:],
        "oi_high": oi_flag.iloc[warmup_c:],
        "funding_rate": df_c["funding_rate"].iloc[warmup_c:],
        "basis_bps": df_c["basis_bps"].iloc[warmup_c:],
    }).fillna(0))

    mask = y.notna()
    try:
        res = QuantReg(y[mask], X[mask]).fit(q=tau, vcov="robust",
                                              kernel="epa", bandwidth="hsheather",
                                              max_iter=5000)
        sensitivity_results.append({
            "test": f"OI_P{oi_pctile}",
            "beta_shock": res.params.get("shock", np.nan),
            "pval_shock": res.pvalues.get("shock", np.nan),
            "beta_interact": res.params.get("shock_x_oi", np.nan),
            "pval_interact": res.pvalues.get("shock_x_oi", np.nan),
        })
    except Exception as e:
        print(f"  Warning OI P{oi_pctile}: {e}")

# ── Alternative variable: collateral seized ──
shock_coll = np.log1p(df_c["liq_usd_collateral"]).shift(1)
oi_flag_80 = (df_c["oi"].rolling(720).rank(pct=True) > 0.80).astype(int)
interaction_coll = shock_coll * oi_flag_80

y = df_c["ret_eth_perp"].iloc[warmup_c:]
X = sm.add_constant(pd.DataFrame({
    "shock": shock_coll.iloc[warmup_c:],
    "shock_x_oi": interaction_coll.iloc[warmup_c:],
    "oi_high": oi_flag_80.iloc[warmup_c:],
    "funding_rate": df_c["funding_rate"].iloc[warmup_c:],
    "basis_bps": df_c["basis_bps"].iloc[warmup_c:],
}).fillna(0))

mask = y.notna()
try:
    res = QuantReg(y[mask], X[mask]).fit(q=tau, vcov="robust",
                                          kernel="epa", bandwidth="hsheather",
                                          max_iter=5000)
    sensitivity_results.append({
        "test": "Collateral_USD",
        "beta_shock": res.params.get("shock", np.nan),
        "pval_shock": res.pvalues.get("shock", np.nan),
        "beta_interact": res.params.get("shock_x_oi", np.nan),
        "pval_interact": res.pvalues.get("shock_x_oi", np.nan),
    })
except Exception as e:
    print(f"  Warning Collateral: {e}")

df_sens = pd.DataFrame(sensitivity_results)

print("=" * 62)
print("TEST C: Sensitivity to Specification Choices (tau=0.01, h=0)")
print("=" * 62)
print(f"{'Specification':<20} {'beta':>10} {'p-val':>8} {'beta_int':>10} {'p-val':>8}")
print("-" * 62)
for _, row in df_sens.iterrows():
    sig = "***" if row["pval_shock"] < 0.01 else ("**" if row["pval_shock"] < 0.05 else "")
    print(f"{row['test']:<20} {row['beta_shock']:>10.4f} {row['pval_shock']:>7.4f}{sig:>3} "
          f"{row['beta_interact']:>10.4f} {row['pval_interact']:>8.4f}")

all_negative = (df_sens["beta_shock"] < 0).all()
range_beta = df_sens["beta_shock"].max() - df_sens["beta_shock"].min()
print(f"\nAll beta(shock) negative: {'YES' if all_negative else 'NO'}")
print(f"Range: {range_beta:.4f}")

TEST C: Sensitivity to Specification Choices (tau=0.01, h=0)
Specification              beta    p-val   beta_int    p-val
--------------------------------------------------------------
OI_P70                  -0.1454  0.0000***    -0.0453   0.1056
OI_P80                  -0.1493  0.0000***    -0.0402   0.1294
OI_P90                  -0.1515  0.0000***    -0.0262   0.3525
Collateral_USD          -0.1484  0.0000***    -0.0398   0.1332

All beta(shock) negative: YES
Range: 0.0062


In [19]:
# ── Save all results ──

# Placebo
df_placebo.to_parquet(ECON_DIR / "robustness_placebo.parquet", index=False, engine="pyarrow")
df_placebo.to_csv(ECON_DIR / "robustness_placebo.csv", index=False)

# Bootstrap
boot_df = pd.DataFrame(boot_results).T
boot_df.index.name = "h"
boot_df.to_csv(ECON_DIR / "robustness_bootstrap.csv")

# Sensitivity
df_sens.to_csv(ECON_DIR / "robustness_sensitivity.csv", index=False)

# QA
qa = {
    "placebo_estimates": len(df_placebo),
    "bootstrap_replications": N_BOOT,
    "bootstrap_block_size": CFG.ECON.block_boot_size,
    "sensitivity_specs": len(df_sens),
    "all_beta_negative": bool(all_negative),
    "status": "PASS" if all_negative else "CHECK",
}
with open(REPORTS_DIR / "robustness_qa.json", "w") as f:
    json.dump(qa, f, indent=2)

print("Saved:")
for f in ["robustness_placebo.parquet", "robustness_placebo.csv",
          "robustness_bootstrap.csv", "robustness_sensitivity.csv"]:
    print(f"  {ECON_DIR / f}")
print(f"  {REPORTS_DIR / 'robustness_qa.json'}")
print("\nNotebook 08 complete")

Saved:
  ./data/econ/robustness_placebo.parquet
  ./data/econ/robustness_placebo.csv
  ./data/econ/robustness_bootstrap.csv
  ./data/econ/robustness_sensitivity.csv
  ./data/analysis/reports/robustness_qa.json

Notebook 08 complete
